In [1]:
import pandas as pd
import json

# load the data

In [12]:
concept_root = "../data/concept/"
out_concept_root = "../data/outside_concept/"
response_root = "../data/respondent/"

In [13]:
# take the concepts 
with open(concept_root + 'new_cid_concept_us_food.json', 'r') as f:
    food_concepts = json.load(f)

# all concepts 
all_us_food_concepts = pd.read_excel(out_concept_root + '0407_cleaned_us_food_concepts.xlsx')

# open transformed
with open(response_root + 'transformed_0407_id_normal_interview.json', 'r', encoding='utf-8') as f:
    transformed_respondent = json.load(f)

# the similarity function

give a function/ the code, which is the similarity search 

input:
content to search, list of content to be searched, top_n （n most relevant ones）, bottom_m (m least relevant ones)

output: 
top n most relevant 
bottom m least relevant  


the algo should be optimal (I don't mind do the embedding for all the available input first ), be fast. 
use "sentence-transformers/all-MiniLM-L6-v2"


In [ ]:
import sys
sys.path.append("../")
from models import similar as sm

In [81]:
# Example usage with caching:
corpus = all_us_food_concepts['ConceptText'].tolist()

# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"

searcher = sm.SimilaritySearcher()
searcher.fit(corpus, cache_path=CACHE_PATH)  # Embeddings cached to disk

# Search (fast - only query embedding computed)
query = "OIKOS TRIPLE ZERO PLAIN HIGH PROTEIN YOGURT\n\nMore of what you want, less of what you don't - 18g of protein 0% fat, 0g added sugars and 0 artificial sweeteners.\n\nAvailable in 32oz large size, Oikos Triple Zero Plain is a deliciously simple way to get the protein you need. Perfect to add to smoothies, parfaits, or enjoy on it's own!\n\n- 18g Protein\n- 0g Added Sugar\n- 0 Artificial Sweeteners\n- 0% Fat\n- Project Non-GMO Verified\n\n32oz Multi-Serve Tub - $5.99\n\nCurrent Oikos Assortment Still Available"
top_results, bottom_results = searcher.search(query, top_n=10, bottom_m=10)

print("Top 5 most similar:")
for text, score in top_results:
    print(f"  {score:.4f}: {text[:80]}...")

print("\nBottom 5 least similar:")
for text, score in bottom_results:
    print(f"  {score:.4f}: {text[:80]}...")

Loading embeddings from cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 8037 embeddings from cache
Top 5 most similar:
  0.9960: Oikos Triple Zero Plain High Protein Yogurt More of what you want, less of what ...
  0.9960: Oikos Triple Zero Plain High Protein Yogurt More of what you want, less of what ...
  0.9369: Oikos Anything but Plain High Protein Yogurt More of what you want, less of what...
  0.9369: Oikos Anything but Plain High Protein Yogurt More of what you want, less of what...
  0.8461: OIKOS Triple Zero Mocha Flavored Yogurt STRONGER MAKES EVERYTHING BETTER® OIKOS ...
  0.8185: OIKOS PRO+ FUEL HIGH-PROTEIN YOGURT WITH COMPLEX CARBS TO FUEL YOU Try new Oikos...
  0.8185: OIKOS PRO+ FUEL HIGH-PROTEIN YOGURT WITH COMPLEX CARBS TO FUEL YOU Try new Oikos...
  0.8113: OIKOS PRO BI-LAYER HIGH-PROTEIN YOGURT WITH A DELIGHTFUL CREAM TO REWARD YOUR EF...
  0.8113: OIKOS PRO BI-LAYER HIGH-PROTEIN YOGURT WITH A DELIGHTFUL CREAM TO REWARD YOUR EF...
  0.8108: OIKOS PRO+ SU

# people like or agains 

In [19]:
import pandas as pd
import json
import os

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI  # Requires langchain-openai package
from dotenv import load_dotenv
load_dotenv()
lite_llm_key_all = os.getenv('LITE_LLM_KEY_ALL')

llm_model = ChatOpenAI(base_url='https://ipsos.litellm-prod.ai/',model='gpt-5', temperature=0.1, api_key=lite_llm_key_all)

In [20]:
from tqdm.asyncio import tqdm
import asyncio
from concurrent.futures import ThreadPoolExecutor


# async multi-thread processing         
def async_multi_tqdm_call(llm_func):
    async def wrapper(input_list, *args, **kargs):
        loop = asyncio.get_event_loop()
        with ThreadPoolExecutor() as pool:
            tasks = [
                loop.run_in_executor(pool, llm_func, input_, *args, **kargs)
                for input_ in input_list
            ]
            # Use tqdm asyncio wrapper around gather
            result = await tqdm.gather(*tasks, desc="processing")
        return result
    return wrapper

1. giving a concept
2. it's needs, chain of thought, would u think he will like it ?  
3. it shall be able to process a list 
that's it 

In [32]:
item = transformed_respondent['101daf40-ed83-11ee-905c-7d576dd0d5d9']

In [40]:
qneeds = [item['qneed2'], item['qneed3']]

In [41]:
qneeds = [f"{q['cate']}: {q['comment']}" for q in qneeds]

In [83]:
from models import need_filter as nf

In [76]:
import random
# random individual 
random.seed(42)  # For reproducibility
n = 10
ids = list(transformed_respondent.keys()); selected_ids = random.sample(ids, n)
# kpi 
kpis = random.choices(["relevance", "differentiation", "believability"], k=n)

# concepts 
concepts = random.choices(all_us_food_concepts['ConceptText'].tolist(), k=n)

In [ ]:


# Run evaluations and collect results
results = {
    'respondent_id': [],
    'respondent_needs': [],
    'kpi_type': [],
    'concept': [],
    'predicted_answer': [],
    'reasoning': []
}

for i, (resp_id, kpi, concept) in enumerate(zip(selected_ids, kpis, concepts)):
    print(f"Processing {i+1}/{n}...")
    respondent_info = transformed_respondent[resp_id]
    
    # Get qneeds for display
    qneeds = []
    for q in ['qneed2', 'qneed3']:
        if q in respondent_info:
            qneeds.append(f"{respondent_info[q]['cate']}: {respondent_info[q]['comment']}")
    qneeds_text = "\n".join(qneeds)
    
    # Run AI filter with reasoning
    reasoning = True
    result = nf.ai_filter(concept, kpi, respondent_info, return_reasoning=reasoning)
    
    results['respondent_id'].append(resp_id)
    results['respondent_needs'].append(qneeds_text)
    results['kpi_type'].append(kpi)
    results['concept'].append(concept[:500])  # truncate for readability
    if reasoning:
        results['predicted_answer'].append(result['answer'])
        results['reasoning'].append(result['reasoning'])
    else:
        results['predicted_answer'].append(result)
        results['reasoning'].append('')

print("Done!")

# Create DataFrame and export to Excel
results_df = pd.DataFrame(results)
output_path = "../data/sample_ai_filter_results_reasoning.xlsx"
results_df.to_excel(output_path, index=False)
print(f"Results saved to {output_path}")

Processing 1/10...
Processing 2/10...


'no'

In [72]:
print(transformed_respondent['40f4c6a0-574b-11ec-a9a8-6b3c43c37c02']['insight'])

**Consumer Needs:** The participant prioritizes health and nutrition, seeking foods with low sugar, high protein, and no artificial ingredients. They value convenience, preferring meals that require minimal preparation. Additionally, they desire variety and enjoyable flavors, particularly in savory dishes.

**Factors of Delight and Differentiation:** The participant finds delight in foods that offer satisfying textures and flavors, such as crunchy items and creamy soups. They appreciate new flavors and are adventurous in trying new packaged foods, driven by curiosity about convenience and taste. However, they maintain a level of skepticism regarding product claims, questioning flavor authenticity and ingredient quality. This balance of adventurousness and caution highlights a demand for innovative yet trustworthy food options.


In [48]:
# Example usage:
concept = all_us_food_concepts['ConceptText'][0]
respondent_info = transformed_respondent['101daf40-ed83-11ee-905c-7d576dd0d5d9']


# With reasoning
result_with_reasoning = ai_filter(concept, "relevance", respondent_info, return_reasoning=True)
print(f"\nAnswer: {result_with_reasoning['answer']}")
print(f"Reasoning: {result_with_reasoning['reasoning']}")



Relevance: no

Answer: no
Reasoning: 1) My needs: low calories/sodium, higher protein, natural ingredients, and small/one- to two-serving packs. I lean toward sweet treats for shelf-stable snacks and get annoyed by big multi-serving bags.
2) The chips have a short, natural ingredient list and an oat-based twist (unique, Trader Joe’s-like), which I like. However, they’re likely higher in calories due to oils, don’t provide meaningful protein, and sodium isn’t specified. The 5oz bag is a multi-serving format I try to avoid.
3) While I’m open to trying unique products, this doesn’t strongly fit my post-gastric sleeve priorities (protein-forward, portion-controlled, low sodium/cal), and it’s not a sweet treat.
